# Document Question Answering System (RAG)

This notebook builds a Retrieval-Augmented Generation pipeline: load a document, split it into chunks, embed the chunks, store them in a vector database, retrieve relevant chunks for a question, and generate an answer using a language model.

## Installing Dependencies

In [1]:
# Install the core libraries used to build the RAG pipeline
!pip install langchain
!pip install langchain_community
!pip install faiss-cpu
!pip install transformers
!pip install langchain_huggingface
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 68.0 MB/s eta 0:00:00


In [2]:
# Install the text splitter package used for chunking documents
!pip install langchain_text_splitters

In [3]:
# Install packages needed to load PDF and DOCX files
!pip install -q langchain langchain-community pypdf docx2txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 13.3 MB/s eta 0:00:00


## Importing Required Libraries

In [4]:

from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_2119/4131373623.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader


## Document Loading Function

In [5]:
def load_documents(file_name):
    # Choose the correct loader based on the file extension
    if file_name.endswith(".pdf"):
        loader = PyPDFLoader(file_name)
    elif file_name.endswith(".txt"):
        loader = TextLoader(file_name)
    elif file_name.endswith(".docx"):
        loader = Docx2txtLoader(file_name)
    else:
        # Stop if the file type is not supported
        raise ValueError(
            f"Unsupported file extension: {file_name}. Use .txt, .pdf, or .docx"
        )

    # Load the document into memory
    document = loader.load()
    return document

## Text Chunking Function

In [6]:
def vector_store(document):
  # Split the document into overlapping chunks for better retrieval accuracy
  splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=52)
  chunks = splitter.split_documents(document)
  # Tag each chunk with an id so it can be referenced later
  for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
  return chunks

## Embedding Model

In [20]:
# Embedding model used to convert text chunks into vectors
embedding_model_1=HuggingFaceEmbeddings(
      model_name="sentence-transformers/all-MiniLM-L12-v2"
      )
embedding_model_2=HuggingFaceEmbeddings(
      model_name="BAAI/bge-small-en-v1.5"
      )
embedding_model_3=HuggingFaceEmbeddings(
      model_name="BAAI/bge-large-en-v1.5"
      )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

## Building the Vector Store

In [8]:
def build_vectorstore(chunks,embedding_model,index_name):
  # Create a FAISS vector store from the document chunks
  vectorstore=FAISS.from_documents(documents=chunks,
                                   embedding=embedding_model
                                   )
  # Save the vector store locally so it can be reloaded later
  vectorstore.save_local(index_name)
  return vectorstore

## Loading the Vector Store

In [9]:
def load_vectorstore(index_name,embedding_model):
    # Load a previously saved FAISS vector store from disk
    vectorstore=FAISS.load_local(index_name,
                                 embedding_model,
                                 allow_dangerous_deserialization=True
                                 )
    return vectorstore

## Running the Pipeline: Load, Chunk, and Embed the Document

In [23]:
# Ask the user for the document to load
file_name=input("Enter the file name:")


document=load_documents(file_name)
chunks=vector_store(document)
print(f"Loaded document and split into {len(chunks)} chunks\n")


print("Different Embedding Models are loading")

db_store_1=build_vectorstore(chunks,embedding_model_1,"faiss_index_minilm")
bd_store_2=build_vectorstore(chunks,embedding_model_2,"faiss_index_bge")
bd_store_3=build_vectorstore(chunks,embedding_model_3,"faiss_index_bge_large")

db_store=load_vectorstore("faiss_index_minilm",embedding_model_1)
print("Vector stores built: faiss_index_minilm (default) and faiss_index_bge, faiss_index_bge_large (for comparison).")

Enter the file name:sample.pdf
Loaded document and split into 883 chunks

Different Embedding Models are loading
Vector stores built: faiss_index_minilm (default) and faiss_index_bge, faiss_index_bge_large (for comparison).


## Loading the Language Model

In [11]:
# Language model used to generate answers from the retrieved context
from langchain_community.llms import HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    device_map="auto",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
        "return_full_text": False,
    },
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Answer Generation Function

In [29]:
def generate_answer(query, retrieved_docs):
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    prompt = f"""<|system|>
You are a helpful AI assistant for document question answering.
Use only the information provided in the context below to answer the question.
Rules:
- Answer only from the provided context.
- Do not make up or assume any information.
- If the answer is not found in the context, reply: "I couldn't find that information in the document."
- Keep your answer clear and concise.<|end|>
<|user|>
Context:
{context}

Question:
{query}<|end|>
<|assistant|>
"""

    # Ask the language model to generate a response
    raw_output = llm.invoke(prompt)

    if "<|assistant|>" in raw_output:
        answer = raw_output.split("<|assistant|>")[-1]
    else:
        answer = raw_output

    # Cut off anything the model rambles into after its actual answer
    for stop_token in ["<|end|>", "<|user|>", "<|system|>", "Question:"]:
        if stop_token in answer:
            answer = answer.split(stop_token)[0]

    return answer.strip()

## Ask a Question

In [13]:
# Get the user's question
query=input("enter the question:")

enter the question:Context of the pdf?


## Retrieve Relevant Chunks

In [14]:
# Retrieve the most relevant chunks for the query
retrieved_docs = db_store.similarity_search(query, k=5)

print(f"Retrieved {len(retrieved_docs)} chunks:\n")
# Preview each retrieved chunk
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {doc.metadata.get('chunk_id')} ---")
    print(doc.page_content[:200], "...\n")

Retrieved 5 chunks:

--- Chunk 458 ---
protection, and financial recompense to rebuff external threats and large-scale industrial monocropping or pastureland 
in which there is very little agrobiodiversity and land degradation may be incre ...

--- Chunk 283 ---
more recent Agreement on Resource Mobilisation and Financial Mechanism in February 2025. Reaching a consensus 
on resource mobilisation and financial mechanisms required a substantial level of comprom ...

--- Chunk 500 ---
several different financial mechanisms might be further developed to fit distinct agricultural contexts. It is this that we 
explore in Section 4. ...

--- Chunk 418 ---
dedicated to the conservation, exchange and revitalisation of native and traditional seeds. It promotes agroecological 
practices and supports biodiversity and food sovereignty through collaborative w ...

--- Chunk 499 ---
reasons of scale and uniformity, or conserve multiple landraces of traditional crop varieties for the same reason. What

## Generate the Answer

In [15]:
# Generate the answer using the retrieved chunks
answer = generate_answer(query, retrieved_docs)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, 

## Display the Answer

In [16]:
# Print the final answer
print(answer)

The context provided is from a document discussing the importance of protecting agrobiodiversity and addressing land degradation. It mentions the need for financial mechanisms to support conservation, exchange, and revitalization of native and traditional seeds, as well as promoting agroecological practices and biodiversity. The document also refers to an Agreement on Resource Mobilization and Financial Mechanism from February 2025, which required compromise to reach a consensus on resource mobilization and financial mechanisms. The document explores different financial mechanisms that could fit distinct agricultural contexts in Section 4.


# Improvements & Experiments

The cells below add the experiments and improvements suggested in the project doc, on top of the pipeline above. Nothing in the existing cells is changed.

In [17]:
# Install packages needed for hybrid search (BM25) and re-ranking (cross-encoder)
!pip install -q rank_bm25 sentence-transformers

## 1. Better chunking strategies

Compare the original fixed-size chunking (300/52) with a token-based splitter and a larger-context splitter.

In [18]:
# Import a token-based text splitter (alternative chunking strategy)
from langchain_text_splitters import TokenTextSplitter

def chunk_token_based(document, chunk_size=256, chunk_overlap=32):
    # Split by token count rather than by character count
    splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    new_chunks = splitter.split_documents(document)
    for i, chunk in enumerate(new_chunks):
        chunk.metadata["chunk_id"] = i
    return new_chunks

def chunk_large_context(document, chunk_size=600, chunk_overlap=100):
    # Use bigger chunks so more context stays together
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    new_chunks = splitter.split_documents(document)
    for i, chunk in enumerate(new_chunks):
        chunk.metadata["chunk_id"] = i
    return new_chunks

# Compare how many chunks each strategy produces
chunking_strategies = {
    "original (300/52)": chunks,
    "token_based (256/32)": chunk_token_based(document),
    "large_context (600/100)": chunk_large_context(document),
}

for name, ch in chunking_strategies.items():
    print(f"{name}: {len(ch)} chunks")

original (300/52): 883 chunks
token_based (256/32): 237 chunks
large_context (600/100): 407 chunks


## 2. Different embedding models

Already implemented above — the notebook builds and compares three embedding models (`all-MiniLM-L12-v2`, `bge-small-en-v1.5`, `bge-large-en-v1.5`). No changes needed here.

In [24]:
# Compare retrieval results across the three embedding-model vector stores
# (db_store_1, bd_store_2, bd_store_3 were built earlier when the document was loaded)

embedding_comparison_query = input("Enter a question to compare embedding models:")

stores_to_compare = {
    "all-MiniLM-L12-v2": db_store_1,
    "bge-small-en-v1.5": bd_store_2,
    "bge-large-en-v1.5": bd_store_3,
}

for model_name, store in stores_to_compare.items():
    print(f"=== {model_name} ===")
    results = store.similarity_search(embedding_comparison_query, k=3)
    for doc in results:
        print(f"--- Chunk {doc.metadata.get('chunk_id')} ---")
        print(doc.page_content[:200], "...\n")
    print()

Enter a question to compare embedding models:Context of the pdf?
=== all-MiniLM-L12-v2 ===
--- Chunk 458 ---
protection, and financial recompense to rebuff external threats and large-scale industrial monocropping or pastureland 
in which there is very little agrobiodiversity and land degradation may be incre ...

--- Chunk 283 ---
more recent Agreement on Resource Mobilisation and Financial Mechanism in February 2025. Reaching a consensus 
on resource mobilisation and financial mechanisms required a substantial level of comprom ...

--- Chunk 500 ---
several different financial mechanisms might be further developed to fit distinct agricultural contexts. It is this that we 
explore in Section 4. ...


=== bge-small-en-v1.5 ===
--- Chunk 845 ---
Sievers-Glotzbach, S, Euler, J, Frison, C, Gmeiner, N, Kliem, L, Maze, A and T chersich, J (2021) Beyond the material: 
knowledge aspects in seed communing, Agriculture and Human Values, 38, pp.509–52 ...

--- Chunk 782 ---
Hejnowicz, AP, Kenter,

## 3. Hybrid search (keyword + vector)

Combine BM25 keyword search with the existing FAISS vector retriever using an ensemble retriever.

In [36]:
!pip install -q langchain langchain-classic

In [37]:
# BM25 does keyword-based retrieval; EnsembleRetriever combines it with vector search
from langchain_community.retrievers import BM25Retriever

# Try every location EnsembleRetriever has lived in across LangChain versions
try:
    from langchain.retrievers import EnsembleRetriever
except ModuleNotFoundError:
    try:
        from langchain_classic.retrievers import EnsembleRetriever
    except ModuleNotFoundError:
        from langchain_community.retrievers import EnsembleRetriever

def build_hybrid_retriever(chunks, vectorstore, k=5, vector_weight=0.5):
    # Keyword-based retriever
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = k

    # Vector-based retriever
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    # Combine both retrievers with adjustable weights
    hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[1 - vector_weight, vector_weight],
    )
    return hybrid_retriever

# Build the hybrid retriever using the existing chunks and vector store
hybrid_retriever = build_hybrid_retriever(chunks, db_store, k=5)

## 4. Re-ranking for better relevance

Use a cross-encoder to re-score the hybrid retrieval results and keep only the most relevant chunks.

In [38]:
# Cross-encoder used to re-score retrieved chunks for relevance
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_documents(query, retrieved_docs, top_k=5):
    # Pair the query with each retrieved chunk
    pairs = [[query, doc.page_content] for doc in retrieved_docs]
    # Score each pair for relevance
    scores = reranker.predict(pairs)
    # Sort chunks by relevance score, highest first
    ranked = sorted(zip(scores, retrieved_docs), key=lambda x: x[0], reverse=True)
    return [doc for score, doc in ranked[:top_k]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## 5. Experiment with different language models

Load an additional language model alongside the existing Phi-3-mini model so answers can be compared.

In [39]:
# Load a second language model to compare answers against Phi-3-mini
llm_alt = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    device_map="auto",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
        "return_full_text": False,
    },
)

def generate_answer_with_model(query, retrieved_docs, model):
    # Combine retrieved chunks into context, same approach as generate_answer
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    prompt = f"""<|system|>
You are a helpful AI assistant for document question answering.
Use only the information provided in the context below to answer the question.
Rules:
- Answer only from the provided context.
- Do not make up or assume any information.
- If the answer is not found in the context, reply: "I couldn't find that information in the document."
- Keep your answer clear and concise.<|end|>
<|user|>
Context:
{context}

Question:
{query}<|end|>
<|assistant|>
"""

    # Generate the raw response from the given model
    raw_output = model.invoke(prompt)

    # Keep only the assistant's reply
    if "<|assistant|>" in raw_output:
        answer = raw_output.split("<|assistant|>")[-1]
    else:
        answer = raw_output

    # Trim off anything after the actual answer
    for stop_token in ["<|end|>", "<|user|>", "<|system|>", "Question:"]:
        if stop_token in answer:
            answer = answer.split(stop_token)[0]
    return answer.strip()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Putting it together: hybrid retrieval + re-ranking + comparing LLM answers

In [43]:

query_experiment = input("Enter the question for the improved pipeline:")

# Retrieve chunks using hybrid search, then re-rank them
hybrid_docs = hybrid_retriever.invoke(query_experiment)
reranked_docs = rerank_documents(query_experiment, hybrid_docs, top_k=5)

# Compare answers generated by two different language models
answer_phi3 = generate_answer(query_experiment, reranked_docs)
answer_tinyllama = generate_answer_with_model(query_experiment, reranked_docs, llm_alt)

print("Answer (Phi-3-mini):\n", answer_phi3)
print("\nAnswer (TinyLlama):\n", answer_tinyllama)

Enter the question for the improved pipeline:Context of the pdf?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer (Phi-3-mini):
 The context provided is from a document discussing the current global ex situ conservation system for plant agrobiodiversity. It mentions a critical review by Engels and Ebert (2021), exploring the development of the global system within a political/legal framework and various financial mechanisms. The document also references a recent Agreement on Resource Mobilisation and Financial Mechanism in February 2025, which required compromise to address issues like least protection and external threats to agrobiodiversity. Additionally, it highlights the challenge of building a resilient diverse system due to a lack of public awareness of current vulnerability, even among the most vulnerable to food price changes (Lang et al., 2025).

Answer (TinyLlama):
 The context of the pdf is a critical review of the current global ex situ conservation system for plant agrobiodiversity. The historical development of the system is discussed in the context of the political/legal fram